# WhisperX testing

In [ ]:
# setup python system path if needed
import sys, json
from pathlib import Path

parent_dir = Path.cwd().resolve().parent  # parent = project root
ffmpeg_bin_path = parent_dir / "ffmpeg" / "bin"

sys.path.append(str(parent_dir))
sys.path.append(str(ffmpeg_bin_path))


def print_formatted_sys_path():
    # Rather than deal with the raw output of sys.path, we can use the json module to spit out a nicely formatted object
    formatted_path = json.dumps(sys.path, indent=4)

    # use some colors to make it pretty!
    print("\033[1;34m[SYS.PATH]\033[0m")  # ANSI code for blue for the title
    print("\033[1;32m" + formatted_path + "\033[0m")  # ANSI code for green for paths, then ANSI code for reset


# Call the function to print the formatted sys.path
print_formatted_sys_path()


In [ ]:
#setup

from pydantic_settings import BaseSettings, SettingsConfigDict
import log_config # to override default and use loguru instead
log_config.setup_logging()

from loguru import logger

class NotebookSettings(BaseSettings):
    hf_token: str
    model_config = SettingsConfigDict(env_file='.env', env_file_encoding='utf-8')

settings = NotebookSettings()

#print(f"HF_TOKEN env variable: {settings.hf_token}")


ModuleNotFoundError: No module named 'whisperx'

In [ ]:
# user configurable settings
audio_file = Path("sample_data/en_US/Gene_Hackman_Interview_by_Bob_Lardine.mp4")
model_dir = Path("models/")


In [ ]:
device = "cuda"
#audio_file = "input_example_audio.mp3"
batch_size = 16 # reduce if low on GPU mem
compute_type = "int8" # change to "float16" if needed for accuracy

# 1. Transcribe with original whisper (batched)
#model = whisperx.load_model("small-v2", device, compute_type=compute_type)

# save model to local path (optional)
model = whisperx.load_model("small-v2", device, compute_type=compute_type, download_root=str(model_dir))

audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)
print(result["segments"]) # before alignment

# delete model if low on GPU resources
# import gc; import torch; gc.collect(); torch.cuda.empty_cache(); del model

# 2. Align whisper output
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

print(result["segments"]) # after alignment

# delete model if low on GPU resources
# TODO will likely need this
#import gc; import torch; gc.collect(); torch.cuda.empty_cache(); del model_a

# 3. Assign speaker labels
diarize_model = DiarizationPipeline(use_auth_token=settings.hf_token, device=device)

# add min/max number of speakers if known
diarize_segments = diarize_model(audio)
# diarize_model(audio, min_speakers=min_speakers, max_speakers=max_speakers)

result = whisperx.assign_word_speakers(diarize_segments, result)
print(diarize_segments)
print(result["segments"]) # segments are now assigned speaker IDs
